In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
 
    message = client.messages.create(**params)
    return message.content[0].text


In [3]:
from datetime import datetime, timedelta
from anthropic.types import ToolParam
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%s"): 
    if not date_format:
        raise ValueError("date format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam(
{
  "name": "get_current_datetime",
  "description": "Returns the current date and time as a formatted string. Use this when you need to know the current date, time, or timestamp in a specific format.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls how the datetime is formatted (e.g. '%Y-%m-%d' for date only, '%H:%M:%S' for time only, '%Y-%m-%d %H:%M:%S' for full timestamp). Cannot be empty or null."
      }
    },
    "required": []
  }
}
)

In [4]:
messages = []

messages.append(
    {"role":"user",
     "content": "What is the current time, formatted as HH:MM:SS"

    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user',
  'content': 'What is the current time, formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01HBDwqBsRcjYBDbLX5VhHXR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [5]:
response

Message(id='msg_01PSh6FqzKL5eqRyjzPycDFA', container=None, content=[ToolUseBlock(id='toolu_01HBDwqBsRcjYBDbLX5VhHXR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=666, output_tokens=63, server_tool_use=None, service_tier='standard'))

In [6]:
result = get_current_datetime(**response.content[0].input)

In [7]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": result,
        "is_error": False
    }]
})  

messages

[{'role': 'user',
  'content': 'What is the current time, formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01HBDwqBsRcjYBDbLX5VhHXR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01HBDwqBsRcjYBDbLX5VhHXR',
    'content': '18:39:24',
    'is_error': False}]}]

In [10]:
final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)
print(final_response.content[0].text)

The current time is **18:39:24** (6:39:24 PM).


In [11]:
print(final_response)

Message(id='msg_01KvvMEHC7iGoAiZVfuxrjZX', container=None, content=[TextBlock(citations=None, text='The current time is **18:39:24** (6:39:24 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=746, output_tokens=23, server_tool_use=None, service_tier='standard'))
